# Precompute Robot Dong Features (Colab)

Runs `precompute_robot_dong.py` on Colab T4 to generate an extended NPZ
with FK + Dong stage2 outputs cached alongside the original `q` array.
Stage 1 training (`train_cross_emb.py`) auto-detects the cached fields
and skips runtime FK on the sampled robot batch each step.

**Setup checklist:**
- Runtime set to GPU: `Runtime > Change runtime type > T4 GPU`
- Drive folder `MyDrive/AIST-hand/` contains:
  - `valid_robot_poses_eigengrasp.npz` (input, q-only NPZ)
  - `dex-urdf/` (Shadow Hand URDF)
  - `shadow_hand_right.yaml` (hand config)

**Output**: `MyDrive/AIST-hand/valid_robot_poses_eigengrasp_dong.npz` (~6.4 GB for 10M poses).
Original input NPZ is left untouched.

## 0. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU found. Go to Runtime > Change runtime type and select T4 GPU.')

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Config

Paste your GitHub token (repo scope) so the latest `precompute_robot_dong.py`
and updated `robot_loader.py` are pulled.

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive/AIST-hand')
GITHUB_TOKEN = ''         # paste GitHub token (repo scope)
GITHUB_USER  = 'isapedraza'
REPO_NAME    = 'AIST-hand'

# Input / output
INPUT_NPZ    = DRIVE_ROOT / 'valid_robot_poses_eigengrasp.npz'
OUTPUT_NPZ   = DRIVE_ROOT / 'valid_robot_poses_eigengrasp_dong.npz'
HAND_CONFIG  = DRIVE_ROOT / 'shadow_hand_right.yaml'
DEX_ROOT     = DRIVE_ROOT / 'dex-urdf'
URDF_PATH    = DEX_ROOT / 'robots/hands/shadow_hand/shadow_hand_right.urdf'

# Precompute params
BATCH_SIZE   = 50_000     # FK batch size on T4 (lower if OOM)
LIMIT        = None       # set int to cap poses (debug). None = all 10M.

for label, p in [
    ('INPUT_NPZ',  INPUT_NPZ),
    ('HAND_CONFIG', HAND_CONFIG),
    ('URDF_PATH',  URDF_PATH),
]:
    print(f'{label}: {p}  exists={p.exists()}')
print(f'OUTPUT_NPZ: {OUTPUT_NPZ}  (will be created/overwritten)')

## 3. Clone repo

In [ ]:
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    clone_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    os.system(f'git clone {clone_url} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull')
    print('Already cloned, pulled latest.')

REPO_ROOT = Path(REPO_DIR)
PRECOMPUTE_SCRIPT = REPO_ROOT / 'grasp-model/scripts/precompute_robot_dong.py'
print(f'Repo: {REPO_ROOT}')
print(f'Script exists: {PRECOMPUTE_SCRIPT.exists()}')

## 4. Install dependencies

In [ ]:
import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION  = torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'torch={TORCH_VERSION}, cuda={CUDA_VERSION}')

!pip install -q pytorch-kinematics

print('Done.')

## 5. Inspect input NPZ

In [ ]:
import numpy as np

data = np.load(INPUT_NPZ)
print('keys :', list(data.keys()))
for k in data.files:
    arr = data[k]
    print(f'  {k}: shape={arr.shape} dtype={arr.dtype}')
size_mb = INPUT_NPZ.stat().st_size / 1e6
print(f'size : {size_mb:,.1f} MB')
del data

## 6. Run precompute

Expected wall time on T4: ~10-20 min for 10M poses with `BATCH_SIZE=50000`.
If OOM, lower `BATCH_SIZE` to 20000.

In [ ]:
import subprocess, sys, time

cmd = [
    sys.executable, '-u',
    str(PRECOMPUTE_SCRIPT),
    '--input',       str(INPUT_NPZ),
    '--urdf',        str(URDF_PATH),
    '--hand_config', str(HAND_CONFIG),
    '--output',      str(OUTPUT_NPZ),
    '--batch_size',  str(BATCH_SIZE),
    '--device',      'cuda',
]
if LIMIT is not None:
    cmd += ['--limit', str(LIMIT)]
print('Running:', ' '.join(cmd))

t0 = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
rc = proc.wait()
elapsed = time.time() - t0
print(f'\nexit code: {rc}  elapsed: {elapsed/60:.1f} min')
if rc != 0:
    raise RuntimeError(f'precompute failed (exit {rc})')

## 7. Verify output

In [ ]:
data = np.load(OUTPUT_NPZ)
print('output keys:', list(data.keys()))
for k in data.files:
    arr = data[k]
    if arr.ndim == 0:
        print(f'  {k}: scalar = {arr}')
    else:
        print(f'  {k}: shape={arr.shape} dtype={arr.dtype}')
size_mb = OUTPUT_NPZ.stat().st_size / 1e6
print(f'size       : {size_mb:,.1f} MB')
print(f'joint_labels: {[str(s) for s in data["joint_labels"]]}')
print(f'tip_labels  : {[str(s) for s in data["tip_labels"]]}')
if "hand_config" in data.files:
    print(f'hand_config : {data["hand_config"]}')
if "hand_length" in data.files:
    print(f'hand_length : {float(data["hand_length"]):.6f}')
del data

## 8. Parity check (optional)

Re-runs `RobotLoader.run_fk` + `_dong_run_stage2` on a small subset of poses
and compares cached outputs vs runtime outputs. Expected diff = 0.0 (or
negligible float noise).

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT / 'grasp-model/scripts'))
from robot_loader import RobotLoader, _dong_run_stage2, _load_hand_config

loader = RobotLoader(str(URDF_PATH), device='cuda', valid_poses_path=str(OUTPUT_NPZ))
print('mode dong_cache?', loader._dong_cache is not None)

torch.manual_seed(42)
q1, quats1, labels1, meta1 = loader.sample_dong(64, str(HAND_CONFIG), seed=42)

config = _load_hand_config(str(HAND_CONFIG))
fk = loader.run_fk(q1)
quats2, labels2, meta2 = _dong_run_stage2(fk, config)
hl = loader._get_hand_length(config)
tips2 = meta2['tips'] / hl
chain2 = {f: v / hl for f, v in meta2['chain_positions'].items()}

print('labels equal       :', labels1 == labels2)
print('tip_labels equal   :', meta1['tip_labels'] == meta2['tip_labels'])
print('quats max abs diff :', (quats1 - quats2).abs().max().item())
print('tips  max abs diff :', (meta1['tips'] - tips2).abs().max().item())
for f in meta1['tip_labels']:
    d = (meta1['chain_positions'][f] - chain2[f]).abs().max().item()
    print(f'chain[{f}] max abs diff:', d)

## Done

Now point Stage 1 training to the new NPZ:

```python
VALID_POSES_PATH = DRIVE_ROOT / 'valid_robot_poses_eigengrasp_dong.npz'
```

Training picks it up automatically (RobotLoader detects the cached fields
and switches to `mode=DONG_CACHE`).